In [ ]:
import os
import warnings

# Note we cannot use rasterio *raw* with matplot/folium. The memory req. to load the moderate size datasets are off the chart
# Also impacts on using plotting off this library (mem manage?)
import rasterio
import rasterio.plot
from rasterio.plot import show_hist

# Local streaming really the only solution here
try:
    from localtileserver import get_folium_tile_layer, TileClient
except:
    !pip install localtileserver
try:
    import folium
except:
    !pip install folium
try:
    import geopandas
except:
    !pip install geopandas

In [ ]:
folderADS = "data/e2e8ac21-0de5-4cbc-ad2e-128cfb028003"
filename = "CITiZAN_All_features_published.csv"
adsfile = os.path.join(folderADS, filename)
print(adsfile)

import pandas
dfCIT = pandas.read_csv(adsfile)
dfCIT

In [ ]:
# Convert into a GeoDataFrame
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(dfCIT['Long'], dfCIT['Lat'])]
gdfCIT = geopandas.GeoDataFrame(dfCIT, geometry=geometry, crs="EPSG:4326")  # WGS84
gdfCIT.head()

In [ ]:
ctmap = folium.Map(location=[gdfCIT.iloc[0].Lat, gdfCIT.iloc[0].Long], zoom_start=8)

# Iterate over the row
for _, row in gdfCIT.iterrows():
    folium.Marker(
        location=[row.Lat, row.Long],
        popup=folium.Popup(f"<b>{row.Description}</b>", max_width=200),
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(ctmap)
ctmap